# CSoT'26 - ML in Astronomy - Week 4: CLIP Zero-Shot Lens Hunter (Starter)

**Goal:** Use a frozen, pretrained **CLIP** model to rank Euclid cutouts by how lens-like they look - no training. Then evaluate the ranking the way a real survey would: precision, recall, and ROC, not accuracy.

**Before you begin:**
1. Switch this notebook to a **GPU runtime** (`Runtime -> Change runtime type -> GPU`). Embedding ~1,476 images with CLIP is much faster on the T4.
2. Read [`03-vision-language-models-and-clip.md`](../03-vision-language-models-and-clip.md) and [`04-zero-shot-lens-finding.md`](../04-zero-shot-lens-finding.md).

Replace each `TODO` with working code. **Do not** open the solution until you've genuinely attempted every TODO.

## Step 0 - Install libraries and pick a device

Colab ships PyTorch and matplotlib, but not Hugging Face `datasets`/`transformers`. Install them, then set the portable device string.

In [ ]:
# Run once per Colab session. Quiet to keep the log short.
!pip install -q datasets transformers

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

## Step 1 - Load the Euclid strong-lens dataset

We use the `classification` config of [`mwalmsley/euclid_strong_lens_expert_judges`](https://huggingface.co/datasets/mwalmsley/euclid_strong_lens_expert_judges). Each example has an `image`, a `label` (`0` = not a grade-A/B lens, `1` = grade-A/B lens), and an `id_str`.

In [ ]:
# TODO: load the train and test splits.
#   from datasets import load_dataset
#   ds_train = load_dataset('mwalmsley/euclid_strong_lens_expert_judges', 'classification', split='train')
#   ds_test  = load_dataset('mwalmsley/euclid_strong_lens_expert_judges', 'classification', split='test')
#   print(ds_train)
#   print(ds_test)

## Step 2 - First analysis cell: class balance

Print train/test sizes and the count of each label. **Lenses are rare** - say the imbalance out loud, because it decides every metric you'll use later (see [`04`](../04-zero-shot-lens-finding.md)). A model that predicts 'not a lens' for everything would score high *accuracy* and find *zero* lenses.

In [ ]:
# TODO: print sizes and label counts for train and test.
#   import numpy as np
#   y_train = np.array(ds_train['label']); y_test = np.array(ds_test['label'])
#   print('train:', len(y_train), 'test:', len(y_test))
#   print('train label counts:', np.bincount(y_train))
#   print('test  label counts:', np.bincount(y_test))
#   print('test positive fraction:', y_test.mean())

## Step 3 - Look at the data

Plot a small grid of example **lenses** (label 1) and **non-lenses** (label 0). Knowing what you're hunting makes your prompts and your error analysis far better. (See [`02-strong-lens-morphologies.md`](../02-strong-lens-morphologies.md).)

In [ ]:
# TODO: show a few positives and negatives.
#   pos_idx = [i for i, y in enumerate(ds_test['label']) if y == 1][:4]
#   neg_idx = [i for i, y in enumerate(ds_test['label']) if y == 0][:4]
#   fig, axes = plt.subplots(2, 4, figsize=(12, 6))
#   for ax, i in zip(axes[0], pos_idx):
#       ax.imshow(ds_test[i]['image']); ax.set_title('lens'); ax.axis('off')
#   for ax, i in zip(axes[1], neg_idx):
#       ax.imshow(ds_test[i]['image']); ax.set_title('not lens'); ax.axis('off')
#   plt.tight_layout(); plt.show()

## Step 4 - Load CLIP

Load `openai/clip-vit-base-patch32` and its matching `CLIPProcessor` (the processor does CLIP's own resize + normalisation - always use it). Move the model to the device and put it in `eval()` mode; we never train it.

In [ ]:
# TODO: load the model and processor.
#   from transformers import CLIPModel, CLIPProcessor
#   model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device).eval()
#   processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

## Step 5 - Build a prompt bank and embed it

Write at least **two lens** prompts and **two non-lens** prompts (start from the bank in [`04`](../04-zero-shot-lens-finding.md)). Notice the non-lens prompts deliberately name the decoys (spiral arms!). Embed the prompts once and L2-normalise them.

In [ ]:
# TODO: define prompts and embed them.
#   lens_prompts = [
#       'a strong gravitational lens with an Einstein ring',
#       'a gravitational lens arc next to a galaxy',
#       'multiple images of a background source due to gravitational lensing']
#   nonlens_prompts = [
#       'a normal galaxy without gravitational lensing',
#       'a spiral galaxy with curved arms but no gravitational lens arc',
#       'a smooth elliptical galaxy with no lensing features']
#   all_prompts = lens_prompts + nonlens_prompts
#   text_inputs = processor(text=all_prompts, return_tensors='pt', padding=True).to(device)
#   with torch.no_grad():
#       text_emb = model.get_text_features(**text_inputs)
#   text_emb = F.normalize(text_emb, dim=-1)            # (num_prompts, D)
#   n_lens = len(lens_prompts)

## Step 6 - Embed all test images (in batches)

Loop over the test set in **chunks of 32-64** under `torch.no_grad()`, run the CLIP image encoder, and L2-normalise. Embedding in batches keeps you off the T4's memory limit. Collect one big `(N, D)` tensor of normalised image embeddings.

In [ ]:
# TODO: embed every test image.
#   images = [ex.convert('RGB') for ex in ds_test['image']]
#   img_embs = []
#   batch = 64
#   for start in range(0, len(images), batch):
#       chunk = images[start:start+batch]
#       inputs = processor(images=chunk, return_tensors='pt').to(device)
#       with torch.no_grad():
#           emb = model.get_image_features(**inputs)
#       img_embs.append(F.normalize(emb, dim=-1).cpu())
#   img_emb = torch.cat(img_embs)                       # (N, D)
#   print(img_emb.shape)

## Step 7 - Score each image

With normalised vectors, cosine similarity is just a matrix product. Compute `sims = img_emb @ text_emb.T` (shape `(N, num_prompts)`), then:

`score = mean(similarity to lens prompts) - mean(similarity to non-lens prompts)`

A positive score means 'more lens-like'. This single number is your detector's output.

In [ ]:
# TODO: compute per-image scores.
#   sims = img_emb @ text_emb.cpu().T                   # (N, num_prompts)
#   lens_sim = sims[:, :n_lens].mean(dim=1)
#   nonlens_sim = sims[:, n_lens:].mean(dim=1)
#   scores = (lens_sim - nonlens_sim).numpy()
#   print('score range:', scores.min(), scores.max())

## Step 8 - Evaluate: ROC-AUC + ROC/PR curves

Report **ROC-AUC** (threshold-free ranking quality: 1.0 perfect, 0.5 random), then plot the ROC and precision-recall curves. For an **imbalanced** problem like ours the PR curve is often more honest. Finally pick a threshold and justify it - real surveys favour **precision** because each candidate costs human follow-up time. (See [`04`](../04-zero-shot-lens-finding.md).)

In [ ]:
# TODO: compute and display metrics.
#   from sklearn.metrics import roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay
#   print('ROC-AUC:', roc_auc_score(y_test, scores))
#   RocCurveDisplay.from_predictions(y_test, scores); plt.title('ROC'); plt.show()
#   PrecisionRecallDisplay.from_predictions(y_test, scores); plt.title('Precision-Recall'); plt.show()
#   # Then choose a threshold t and report precision/recall at it, e.g. with classification_report.

## Step 9 - Error gallery

Sort by score and display: the **top true positives** (highest-scoring real lenses), the **worst false positives** (highest-scoring non-lenses - expect spirals/ring galaxies), and the **worst false negatives** (real lenses scored low). Title each panel with its score and true label. This gallery is where the science lives.

In [ ]:
# TODO: build the error gallery.
#   order = np.argsort(-scores)                         # high score first
#   y = np.array(y_test)
#   top_tp = [i for i in order if y[i] == 1][:4]
#   worst_fp = [i for i in order if y[i] == 0][:4]
#   worst_fn = [i for i in order[::-1] if y[i] == 1][:4]
#   # Plot each row with imshow(ds_test[i]['image']) and a title f"{scores[i]:.3f} / true={y[i]}".

## Reflection *(write 2-3 sentences each)*

1. What ROC-AUC did your zero-shot CLIP reach? Given that no labelled lenses were used to build the scorer, is that better or worse than you expected?
2. Pick one of your worst false positives and explain - using [`02-strong-lens-morphologies.md`](../02-strong-lens-morphologies.md) - why a human could reasonably mistake it for a lens.
3. What accuracy would a 'say no to everything' model get on your test set, and why did you report precision/recall instead?

*(Replace this prompt with your answers.)*